# 03 — Construcción de OBT (One Big Table)

Ensambla `NYC_TAXI_P3.ANALYTICS.OBT_TRIPS` a partir de `RAW.TRIPS_RAW` + `TAXI_ZONE_LOOKUP`.

**Grano:** 1 fila = 1 viaje.  
**Contenido:** hechos + zonas desnormalizadas + catálogos decodificados + derivadas + lineage.

**Derivadas calculadas:**
- `trip_duration_min` = DATEDIFF en minutos entre pickup y dropoff.
- `avg_speed_mph` = trip_distance / (duration_min / 60). NULL si duración o distancia ≤ 0.
- `tip_pct` = (tip_amount / fare_amount) × 100. NULL si fare_amount ≤ 0.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("=" * 60)
print("CELDA 1: Inicialización Spark + Configuración Snowflake")
print("=" * 60)

spark = SparkSession.builder.appName("NYC_Taxi_OBT").getOrCreate()
print("✓ SparkSession creada")

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}

SF_OPTS_ANALYTICS = {**SF_OPTIONS, "sfSchema": os.environ["SF_SCHEMA_ANALYTICS"]}

print(f"✓ Snowflake URL: {SF_OPTIONS['sfURL']}")
print(f"✓ Schema RAW: {os.environ['SF_SCHEMA_RAW']}")
print(f"✓ Schema ANALYTICS: {os.environ['SF_SCHEMA_ANALYTICS']}")
print("=" * 60)

CELDA 1: Inicialización Spark + Configuración Snowflake
✓ SparkSession creada
✓ Snowflake URL: AASKQCL-YY64872.snowflakecomputing.com
✓ Schema RAW: RAW
✓ Schema ANALYTICS: ANALYTICS


## 3.1 Consulta OBT

Se ejecuta un SQL pushdown en Snowflake que:
1. Unifica timestamps Yellow/Green con `COALESCE`.
2. Extrae componentes de fecha/hora.
3. Hace JOIN con `TAXI_ZONE_LOOKUP` para pickup y dropoff.
4. Decodifica catálogos (vendor, payment, rate_code).
5. Calcula derivadas (duración, velocidad, % propina).
6. Preserva metadatos de lineage.

In [2]:
print("=" * 60)
print("CELDA 2: Consulta SQL para construir OBT")
print("=" * 60)

obt_sql = """
SELECT
    -- Tiempo unificado
    COALESCE(r.tpep_pickup_datetime,  r.lpep_pickup_datetime)  AS pickup_datetime,
    COALESCE(r.tpep_dropoff_datetime, r.lpep_dropoff_datetime) AS dropoff_datetime,
    TO_DATE(COALESCE(r.tpep_pickup_datetime,  r.lpep_pickup_datetime))  AS pickup_date,
    EXTRACT(HOUR FROM COALESCE(r.tpep_pickup_datetime,  r.lpep_pickup_datetime))  AS pickup_hour,
    TO_DATE(COALESCE(r.tpep_dropoff_datetime, r.lpep_dropoff_datetime)) AS dropoff_date,
    EXTRACT(HOUR FROM COALESCE(r.tpep_dropoff_datetime, r.lpep_dropoff_datetime)) AS dropoff_hour,
    DAYOFWEEK(COALESCE(r.tpep_pickup_datetime, r.lpep_pickup_datetime)) AS day_of_week,
    r.source_month                        AS month,
    r.source_year                         AS year,

    -- Ubicación enriquecida
    r.PULocationID                        AS pu_location_id,
    pz.Zone                               AS pu_zone,
    pz.Borough                            AS pu_borough,
    r.DOLocationID                        AS do_location_id,
    dz.Zone                               AS do_zone,
    dz.Borough                            AS do_borough,

    -- Servicio y catálogos decodificados
    r.service_type,
    r.VendorID                            AS vendor_id,
    CASE r.VendorID
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'VeriFone Inc.'
        ELSE 'Unknown'
    END                                   AS vendor_name,
    r.RatecodeID::INTEGER                 AS rate_code_id,
    CASE r.RatecodeID::INTEGER
        WHEN 1 THEN 'Standard rate'
        WHEN 2 THEN 'JFK'
        WHEN 3 THEN 'Newark'
        WHEN 4 THEN 'Nassau/Westchester'
        WHEN 5 THEN 'Negotiated fare'
        WHEN 6 THEN 'Group ride'
        ELSE 'Unknown'
    END                                   AS rate_code_desc,
    r.payment_type,
    CASE r.payment_type
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
        ELSE 'Other'
    END                                   AS payment_type_desc,
    r.trip_type,

    -- Viaje
    r.passenger_count,
    r.trip_distance,
    r.store_and_fwd_flag,

    -- Tarifas
    r.fare_amount,
    r.extra,
    r.mta_tax,
    r.tip_amount,
    r.tolls_amount,
    r.improvement_surcharge,
    r.congestion_surcharge,
    r.Airport_fee                         AS airport_fee,
    r.total_amount,

    -- Derivadas
    DATEDIFF('minute',
        COALESCE(r.tpep_pickup_datetime, r.lpep_pickup_datetime),
        COALESCE(r.tpep_dropoff_datetime, r.lpep_dropoff_datetime)
    )                                     AS trip_duration_min,
    CASE
        WHEN DATEDIFF('minute',
            COALESCE(r.tpep_pickup_datetime, r.lpep_pickup_datetime),
            COALESCE(r.tpep_dropoff_datetime, r.lpep_dropoff_datetime)) > 0
         AND r.trip_distance > 0
        THEN ROUND(r.trip_distance / (DATEDIFF('minute',
            COALESCE(r.tpep_pickup_datetime, r.lpep_pickup_datetime),
            COALESCE(r.tpep_dropoff_datetime, r.lpep_dropoff_datetime)) / 60.0), 2)
        ELSE NULL
    END                                   AS avg_speed_mph,
    CASE
        WHEN r.fare_amount > 0
        THEN ROUND((r.tip_amount / r.fare_amount) * 100, 2)
        ELSE NULL
    END                                   AS tip_pct,

    -- Lineage
    r.run_id,
    r.ingested_at_utc,
    r.service_type                        AS source_service,
    r.source_year,
    r.source_month

FROM NYC_TAXI_P3.RAW.TRIPS_RAW r
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP pz ON r.PULocationID = pz.LocationID
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP dz ON r.DOLocationID = dz.LocationID
"""

print(">>> Ejecutando SQL pushdown para construir OBT...")
df_obt = (spark.read
    .format("snowflake")
    .options(**SF_OPTIONS)
    .option("query", obt_sql)
    .load())

row_count = df_obt.count()
print(f"✓ OBT construida: {row_count:,} filas")
df_obt.printSchema()
print("=" * 60)

CELDA 2: Consulta SQL para construir OBT
>>> Ejecutando SQL pushdown para construir OBT...
✓ OBT construida: 852,877,437 filas
root
 |-- PICKUP_DATETIME: timestamp (nullable = true)
 |-- DROPOFF_DATETIME: timestamp (nullable = true)
 |-- PICKUP_DATE: date (nullable = true)
 |-- PICKUP_HOUR: decimal(2,0) (nullable = true)
 |-- DROPOFF_DATE: date (nullable = true)
 |-- DROPOFF_HOUR: decimal(2,0) (nullable = true)
 |-- DAY_OF_WEEK: decimal(2,0) (nullable = true)
 |-- MONTH: decimal(38,0) (nullable = false)
 |-- YEAR: decimal(38,0) (nullable = false)
 |-- PU_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- PU_ZONE: string (nullable = true)
 |-- PU_BOROUGH: string (nullable = true)
 |-- DO_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- DO_ZONE: string (nullable = true)
 |-- DO_BOROUGH: string (nullable = true)
 |-- SERVICE_TYPE: string (nullable = false)
 |-- VENDOR_ID: decimal(38,0) (nullable = true)
 |-- VENDOR_NAME: string (nullable = false)
 |-- RATE_CODE_ID: decimal(38,0) (nullable

## 3.2 Escribir OBT en Snowflake (overwrite = idempotente)

In [3]:
print("=" * 60)
print("CELDA 3: Escribir OBT en Snowflake (overwrite = idempotente)")
print("=" * 60)

print(">>> Escribiendo OBT_TRIPS en ANALYTICS...")
(df_obt.write
    .format("snowflake")
    .options(**SF_OPTS_ANALYTICS)
    .option("dbtable", "OBT_TRIPS")
    .mode("overwrite")
    .save())

print("✓ OBT_TRIPS escrita exitosamente en ANALYTICS")
print("=" * 60)

CELDA 3: Escribir OBT en Snowflake (overwrite = idempotente)
>>> Escribiendo OBT_TRIPS en ANALYTICS...


Py4JJavaError: An error occurred while calling o47.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 13 in stage 3.0 failed 1 times, most recent failure: Lost task 13.0 in stage 3.0 (TID 214) (e0380c065433 executor driver): net.snowflake.client.jdbc.SnowflakeSQLLoggedException: JDBC driver internal error: Max retry reached for the download of #chunk0 (Total chunks: 1) retry=7, error=net.snowflake.client.jdbc.SnowflakeSQLLoggedException: JDBC driver internal error: Exception: Failure allocating buffer..
	at net.snowflake.client.jdbc.SnowflakeChunkDownloader$2.downloadAndParseChunk(SnowflakeChunkDownloader.java:947)
	at net.snowflake.client.jdbc.SnowflakeChunkDownloader$2.call(SnowflakeChunkDownloader.java:1002)
	at net.snowflake.client.jdbc.SnowflakeChunkDownloader$2.call(SnowflakeChunkDownloader.java:913)
	at java.base/java.util.concurrent.FutureTask.run(FutureTask.java:264)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor

## 3.3 Verificación de idempotencia

Reejecutamos la construcción de OBT y verificamos que no se dupliquen filas.

In [ ]:
print("=" * 60)
print("CELDA 4: Verificación de idempotencia")
print("=" * 60)

# Contar filas antes
count_before_sql = "SELECT COUNT(*) AS cnt FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS"
rows_before = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", count_before_sql).load()
    .collect()[0][0])
print(f">>> Filas antes de rebuild: {rows_before:,}")

# Rebuild OBT (overwrite)
print(">>> Re-ejecutando OBT (overwrite)...")
df_obt2 = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", obt_sql).load())
(df_obt2.write.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("dbtable", "OBT_TRIPS")
    .mode("overwrite").save())

# Contar filas después
rows_after = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", count_before_sql).load()
    .collect()[0][0])
print(f">>> Filas después de rebuild: {rows_after:,}")

assert rows_before == rows_after, f"FALLO IDEMPOTENCIA: {rows_before} != {rows_after}"
print(f"✓ Idempotencia verificada: {rows_before:,} == {rows_after:,}")
print("=" * 60)

## 3.4 Muestra de la OBT final

In [ ]:
print("=" * 60)
print("CELDA 5: Muestra de la OBT final")
print("=" * 60)

sample_sql = "SELECT * FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS LIMIT 10"
print(">>> Consultando muestra...")
df_sample = (spark.read.format("snowflake")
    .options(**SF_OPTS_ANALYTICS).option("query", sample_sql).load())
df_sample.show(5, truncate=False)
print("✓ NB03 completado — OBT construida y verificada")
print("=" * 60)